# Baseline Experiments for Multi-Omics Analysis
Implements unimodal VAEs and simple fusion approaches

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, Tuple, Any
import warnings
warnings.filterwarnings('ignore')

# Additional imports for model saving
import pickle
import os
from pathlib import Path
from datetime import datetime

In [2]:
class UnimodalVAE(nn.Module):
    """Variational Autoencoder for single modality"""
    
    def __init__(self, input_dim: int, latent_dim: int = 50, hidden_dims: list = None):
        super(UnimodalVAE, self).__init__()
        
        if hidden_dims is None:
            hidden_dims = [512, 256, 128]
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = hidden_dim
        
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Latent space
        self.mu_layer = nn.Linear(prev_dim, latent_dim)
        self.logvar_layer = nn.Linear(prev_dim, latent_dim)
        
        # Decoder
        decoder_layers = []
        prev_dim = latent_dim
        
        for hidden_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = hidden_dim
        
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
        
    def encode(self, x):
        """Encode input to latent parameters"""
        h = self.encoder(x)
        mu = self.mu_layer(h)
        logvar = self.logvar_layer(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        """Decode latent vector to reconstruction"""
        return self.decoder(z)
    
    def forward(self, x):
        """Forward pass"""
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

In [3]:
def vae_loss_function(recon_x, x, mu, logvar, beta=1.0):
    """VAE loss function with KL divergence"""
    # Reconstruction loss
    recon_loss = nn.MSELoss(reduction='sum')(recon_x, x)
    
    # KL divergence
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

In [ ]:
class BaselineExperiments:
    def __init__(self, data_loader, experiment_tracker):
        self.loader = data_loader
        self.tracker = experiment_tracker
        self.results = {}
        self.models = {}
        
        # Device setup
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
    
    def save_trained_model(self, model, model_name, epoch, optimizer, metrics):
        """Save trained model with experiment tracking"""
        
        # Create models directory in the tracker's models directory
        models_dir = self.tracker.models_dir
        models_dir.mkdir(parents=True, exist_ok=True)
        
        # Generate filename with timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        base_filename = f"sample_{model_name}_{timestamp}"
        
        print(f"💾 Saving sample data model: {base_filename}")
        
        # Save state dictionary (recommended method)
        state_dict_path = models_dir / f"{base_filename}_state.pt"
        
        checkpoint = {
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'model_architecture': {
                'class_name': model.__class__.__name__,
                'input_dim': model.input_dim,
                'latent_dim': model.latent_dim,
            },
            'training_info': {
                'epoch': epoch,
                'timestamp': timestamp,
                'pytorch_version': torch.__version__,
                'dataset_type': 'sample_tcga'
            },
            'final_metrics': metrics
        }
        
        torch.save(checkpoint, state_dict_path)
        
        # Save full model (alternative method)
        full_model_path = models_dir / f"{base_filename}_full.pt"
        torch.save(model, full_model_path)
        
        # Save in pickle format (for compatibility)
        pickle_path = models_dir / f"{base_filename}.pkl"
        with open(pickle_path, 'wb') as f:
            pickle.dump({
                'model': model,
                'checkpoint': checkpoint,
                'saved_timestamp': timestamp
            }, f)
        
        # Register with experiment tracker
        self.tracker.save_artifact(
            str(state_dict_path), "model_state_dict", 
            f"Trained sample {model_name} state dictionary"
        )
        self.tracker.save_artifact(
            str(full_model_path), "model_full", 
            f"Trained sample {model_name} complete model"
        )
        self.tracker.save_artifact(
            str(pickle_path), "model_pickle", 
            f"Trained sample {model_name} pickle format"
        )
        
        self.tracker.log_message(f"Sample model saved successfully: {base_filename}")
        self.tracker.log_message(f"  - State dict: {state_dict_path.name}")
        self.tracker.log_message(f"  - Full model: {full_model_path.name}")
        self.tracker.log_message(f"  - Pickle: {pickle_path.name}")
        
        print(f"✅ Sample model saved to {models_dir}/")
        print(f"   📁 State dict: {state_dict_path}")
        print(f"   📁 Full model: {full_model_path}")
        print(f"   📁 Pickle: {pickle_path}")
        
        return str(state_dict_path)
    
    def prepare_data(self, modality_data: pd.DataFrame, test_size: float = 0.2) -> Tuple[torch.Tensor, torch.Tensor]:
        """Prepare data for training"""
        # Handle missing values
        data_clean = modality_data.fillna(modality_data.mean())
        
        # Standardize features
        scaler = StandardScaler()
        data_scaled = scaler.fit_transform(data_clean.T)  # Transpose for samples x features
        
        # Split data
        train_data, test_data = train_test_split(data_scaled, test_size=test_size, random_state=42)
        
        # Convert to tensors
        train_tensor = torch.FloatTensor(train_data).to(self.device)
        test_tensor = torch.FloatTensor(test_data).to(self.device)
        
        return train_tensor, test_tensor
    
    def train_unimodal_vae(self, 
                          modality_name: str,
                          modality_data: pd.DataFrame,
                          latent_dim: int = 50,
                          epochs: int = 100,
                          batch_size: int = 32,
                          learning_rate: float = 0.001) -> Dict[str, Any]:
        """Train unimodal VAE for single modality - Research Question 5"""
        
        print(f"\n=== Training Sample {modality_name} VAE ===")
        
        # Start experiment tracking
        exp_id = self.tracker.start_experiment(
            experiment_name=f"unimodal_vae_{modality_name.lower()}",
            experiment_type="baseline",
            description=f"Unimodal VAE for sample {modality_name} expression data",
            parameters={
                "modality": modality_name,
                "latent_dim": latent_dim,
                "epochs": epochs,
                "batch_size": batch_size,
                "learning_rate": learning_rate
            },
            research_questions=["How well can each modality individually predict breast cancer subtypes?"]
        )
        
        try:
            # Prepare data
            train_data, test_data = self.prepare_data(modality_data)
            input_dim = train_data.shape[1]
            
            self.tracker.log_message(f"Data prepared: {train_data.shape[0]} train, {test_data.shape[0]} test samples")
            self.tracker.log_message(f"Input dimension: {input_dim}")
            
            # Create model
            model = UnimodalVAE(input_dim=input_dim, latent_dim=latent_dim).to(self.device)
            optimizer = optim.Adam(model.parameters(), lr=learning_rate)
            
            # Create data loader
            train_dataset = TensorDataset(train_data)
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            
            # Training loop
            model.train()
            train_losses = []
            
            for epoch in range(epochs):
                epoch_loss = 0
                epoch_recon_loss = 0
                epoch_kl_loss = 0
                
                for batch_idx, (data,) in enumerate(train_loader):
                    optimizer.zero_grad()
                    
                    recon, mu, logvar = model(data)
                    loss, recon_loss, kl_loss = vae_loss_function(recon, data, mu, logvar)
                    
                    loss.backward()
                    optimizer.step()
                    
                    epoch_loss += loss.item()
                    epoch_recon_loss += recon_loss.item()
                    epoch_kl_loss += kl_loss.item()
                
                # Log metrics
                avg_loss = epoch_loss / len(train_loader)
                avg_recon = epoch_recon_loss / len(train_loader)
                avg_kl = epoch_kl_loss / len(train_loader)
                
                train_losses.append(avg_loss)
                
                if epoch % 20 == 0:
                    self.tracker.log_metric("train_loss", avg_loss, epoch)
                    self.tracker.log_metric("reconstruction_loss", avg_recon, epoch)
                    self.tracker.log_metric("kl_loss", avg_kl, epoch)
                    print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Recon={avg_recon:.4f}, KL={avg_kl:.4f}")
            
            # Evaluation
            model.eval()
            with torch.no_grad():
                test_recon, test_mu, test_logvar = model(test_data)
                test_loss, test_recon_loss, test_kl_loss = vae_loss_function(test_recon, test_data, test_mu, test_logvar)
                
                # Reconstruction accuracy
                mse = nn.MSELoss()(test_recon, test_data).item()
                correlation = np.corrcoef(test_data.cpu().numpy().flatten(), 
                                       test_recon.cpu().numpy().flatten())[0, 1]
            
            # Log final metrics
            self.tracker.log_metric("final_test_loss", test_loss.item())
            self.tracker.log_metric("final_mse", mse)
            self.tracker.log_metric("reconstruction_correlation", correlation)
            
            # Save the trained model
            model_path = self.save_trained_model(
                model=model,
                model_name=f"{modality_name.lower()}_vae",
                epoch=epoch,
                optimizer=optimizer,
                metrics={
                    "final_test_loss": test_loss.item(),
                    "mse": mse,
                    "correlation": correlation
                }
            )
            
            # Create training plot
            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.plot(train_losses)
            plt.title(f'Sample {modality_name} VAE Training Loss')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            
            plt.subplot(1, 2, 2)
            # Plot reconstruction comparison
            sample_idx = 0
            original = test_data[sample_idx].cpu().numpy()
            reconstructed = test_recon[sample_idx].cpu().numpy()
            
            plt.scatter(original, reconstructed, alpha=0.5)
            plt.plot([original.min(), original.max()], [original.min(), original.max()], 'r--')
            plt.xlabel('Original')
            plt.ylabel('Reconstructed')
            plt.title(f'Sample {modality_name} Reconstruction Quality')
            
            plt.tight_layout()
            plot_path = f"sample_{modality_name.lower()}_vae_training.png"
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            self.tracker.save_artifact(plot_path, "plot", f"Sample {modality_name} VAE training visualization")
            plt.show()
            
            # Store results
            results = {
                "model": model,
                "model_path": model_path,
                "final_loss": test_loss.item(),
                "mse": mse,
                "correlation": correlation,
                "latent_dim": latent_dim,
                "input_dim": input_dim,
                "train_losses": train_losses
            }
            
            self.models[f"{modality_name.lower()}_vae"] = model
            self.results[f"{modality_name.lower()}_vae"] = results
            
            self.tracker.end_experiment(status="completed")
            
            print(f"Sample {modality_name} VAE training completed!")
            print(f"Final test loss: {test_loss.item():.4f}")
            print(f"Reconstruction correlation: {correlation:.4f}")
            print(f"Model saved: {model_path}")
            
            return results
            
        except Exception as e:
            self.tracker.record_failed_experiment(
                error_message=str(e),
                error_type=type(e).__name__,
                attempted_fixes=["Check data preprocessing", "Verify model architecture"]
            )
            raise e
    
    def compare_modality_performance(self) -> pd.DataFrame:
        """Research Question 6: Compare individual modality performance"""
        
        print("\n=== Sample Data Modality Performance Comparison ===")
        
        comparison_data = []
        
        for model_name, results in self.results.items():
            if "vae" in model_name:
                modality = model_name.replace("_vae", "")
                comparison_data.append({
                    "modality": modality,
                    "final_loss": results["final_loss"],
                    "mse": results["mse"],
                    "correlation": results["correlation"],
                    "latent_dim": results["latent_dim"],
                    "input_dim": results["input_dim"],
                    "model_path": results.get("model_path", "N/A")
                })
        
        comparison_df = pd.DataFrame(comparison_data)
        
        if not comparison_df.empty:
            # Create comparison plots
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            
            # Loss comparison
            axes[0].bar(comparison_df["modality"], comparison_df["final_loss"])
            axes[0].set_title("Final Test Loss by Modality (Sample Data)")
            axes[0].set_ylabel("Loss")
            axes[0].tick_params(axis='x', rotation=45)
            
            # MSE comparison
            axes[1].bar(comparison_df["modality"], comparison_df["mse"])
            axes[1].set_title("Reconstruction MSE by Modality (Sample Data)")
            axes[1].set_ylabel("MSE")
            axes[1].tick_params(axis='x', rotation=45)
            
            # Correlation comparison
            axes[2].bar(comparison_df["modality"], comparison_df["correlation"])
            axes[2].set_title("Reconstruction Correlation by Modality (Sample Data)")
            axes[2].set_ylabel("Correlation")
            axes[2].tick_params(axis='x', rotation=45)
            
            plt.tight_layout()
            plt.savefig("sample_modality_performance_comparison.png", dpi=300, bbox_inches='tight')
            plt.show()
            
            # Print ranking
            print("\nSample Data Modality Performance Ranking (by reconstruction correlation):")
            ranked = comparison_df.sort_values("correlation", ascending=False)
            for idx, row in ranked.iterrows():
                print(f"{row['modality']}: correlation={row['correlation']:.4f}, loss={row['final_loss']:.4f}")
                print(f"   Model saved: {Path(row['model_path']).name if row['model_path'] != 'N/A' else 'N/A'}")
        
        return comparison_df
    
    def train_simple_fusion(self, modalities_data: Dict[str, pd.DataFrame]) -> Dict[str, Any]:
        """Research Question 8: Simple concatenation fusion baseline"""
        
        print("\n=== Training Sample Simple Fusion Baseline ===")
        
        exp_id = self.tracker.start_experiment(
            experiment_name="simple_concatenation_fusion",
            experiment_type="fusion_baseline",
            description="Simple feature concatenation fusion of sample modalities",
            parameters={
                "fusion_type": "concatenation",
                "modalities": list(modalities_data.keys())
            },
            research_questions=["How does simple concatenation perform compared to individual modalities?"]
        )
        
        try:
            # Concatenate all modalities
            concatenated_features = []
            feature_info = {}
            
            for modality_name, data in modalities_data.items():
                if data is not None:
                    # Standardize and clean
                    data_clean = data.fillna(data.mean())
                    scaler = StandardScaler()
                    data_scaled = scaler.fit_transform(data_clean.T)  # samples x features
                    
                    concatenated_features.append(data_scaled)
                    feature_info[modality_name] = data_scaled.shape[1]
                    
                    self.tracker.log_message(f"Added {modality_name}: {data_scaled.shape[1]} features")
            
            # Concatenate all features
            if concatenated_features:
                all_features = np.concatenate(concatenated_features, axis=1)
                self.tracker.log_message(f"Total concatenated features: {all_features.shape[1]}")
                
                # Split data
                train_data, test_data = train_test_split(all_features, test_size=0.2, random_state=42)
                
                # Convert to tensors
                train_tensor = torch.FloatTensor(train_data).to(self.device)
                test_tensor = torch.FloatTensor(test_data).to(self.device)
                
                # Train fusion VAE
                fusion_model = UnimodalVAE(
                    input_dim=all_features.shape[1], 
                    latent_dim=100,  # Larger latent space for fusion
                    hidden_dims=[1024, 512, 256]  # Larger hidden layers
                ).to(self.device)
                
                optimizer = optim.Adam(fusion_model.parameters(), lr=0.0005)  # Lower learning rate
                
                # Training
                fusion_model.train()
                train_dataset = TensorDataset(train_tensor)
                train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)  # Smaller batch size
                
                for epoch in range(50):  # Fewer epochs due to complexity
                    epoch_loss = 0
                    
                    for batch_idx, (data,) in enumerate(train_loader):
                        optimizer.zero_grad()
                        recon, mu, logvar = fusion_model(data)
                        loss, recon_loss, kl_loss = vae_loss_function(recon, data, mu, logvar)
                        
                        loss.backward()
                        optimizer.step()
                        epoch_loss += loss.item()
                    
                    if epoch % 10 == 0:
                        avg_loss = epoch_loss / len(train_loader)
                        self.tracker.log_metric("fusion_train_loss", avg_loss, epoch)
                        print(f"Sample Fusion Epoch {epoch}: Loss={avg_loss:.4f}")
                
                # Evaluation
                fusion_model.eval()
                with torch.no_grad():
                    test_recon, test_mu, test_logvar = fusion_model(test_tensor)
                    test_loss, _, _ = vae_loss_function(test_recon, test_tensor, test_mu, test_logvar)
                    
                    fusion_mse = nn.MSELoss()(test_recon, test_tensor).item()
                    fusion_correlation = np.corrcoef(test_tensor.cpu().numpy().flatten(),
                                                   test_recon.cpu().numpy().flatten())[0, 1]
                
                # Log final metrics
                self.tracker.log_metric("fusion_final_loss", test_loss.item())
                self.tracker.log_metric("fusion_mse", fusion_mse)
                self.tracker.log_metric("fusion_correlation", fusion_correlation)
                
                # Save the fusion model
                fusion_model_path = self.save_trained_model(
                    model=fusion_model,
                    model_name="fusion_concatenation_vae",
                    epoch=epoch,
                    optimizer=optimizer,
                    metrics={
                        "fusion_final_loss": test_loss.item(),
                        "fusion_mse": fusion_mse,
                        "fusion_correlation": fusion_correlation
                    }
                )
                
                results = {
                    "model": fusion_model,
                    "model_path": fusion_model_path,
                    "final_loss": test_loss.item(),
                    "mse": fusion_mse,
                    "correlation": fusion_correlation,
                    "feature_info": feature_info,
                    "total_features": all_features.shape[1]
                }
                
                self.models["fusion_vae"] = fusion_model
                self.results["fusion_vae"] = results
                
                self.tracker.end_experiment(status="completed")
                
                print(f"Sample Fusion VAE training completed!")
                print(f"Final test loss: {test_loss.item():.4f}")
                print(f"Reconstruction correlation: {fusion_correlation:.4f}")
                print(f"Model saved: {fusion_model_path}")
                
                return results
            
        except Exception as e:
            self.tracker.record_failed_experiment(
                error_message=str(e),
                error_type=type(e).__name__,
                attempted_fixes=["Reduce model complexity", "Use gradient clipping"]
            )
            raise e
    
    def run_all_baseline_experiments(self):
        """Run complete baseline experiment suite"""
        print("Starting Sample Data Baseline Experiments Suite...")
        
        # Get data
        omics_data = {
            "mRNA": self.loader.mrna_data,
            "miRNA": self.loader.mirna_data,
            "Methylation": self.loader.methylation_data,
            "CNV": self.loader.cnv_data
        }
        
        # Remove None data
        omics_data = {k: v for k, v in omics_data.items() if v is not None}
        
        # Train unimodal VAEs
        for modality_name, data in omics_data.items():
            try:
                self.train_unimodal_vae(modality_name, data)
            except Exception as e:
                print(f"Failed to train sample {modality_name} VAE: {e}")
        
        # Compare modalities
        comparison_df = self.compare_modality_performance()
        
        # Train fusion baseline
        try:
            self.train_simple_fusion(omics_data)
        except Exception as e:
            print(f"Failed to train sample fusion model: {e}")
        
        return self.results, comparison_df

In [5]:
# Initialize and run baseline experiments
print("Initializing baseline experiments with real TCGA data...")

# Check if loader and tracker are available
if 'loader' in globals() and 'tracker' in globals():
    # Initialize baseline experiments
    baseline_exp = BaselineExperiments(loader, tracker)

    # Run all experiments
    print("Starting baseline experiments suite...")
    results, comparison = baseline_exp.run_all_baseline_experiments()

    # Print summary
    print("\n" + "="*50)
    print("BASELINE EXPERIMENTS SUMMARY")
    print("="*50)
    if not comparison.empty:
        print(comparison)
        print(f"\nBest performing modality: {comparison.loc[comparison['correlation'].idxmax(), 'modality']}")
    else:
        print("No successful experiments completed")
        
    print("✓ Baseline experiments completed successfully!")
    
    # Ensure variables are available globally for main pipeline
    globals()['baseline_exp'] = baseline_exp
    globals()['results'] = results
    globals()['comparison'] = comparison
    
else:
    print("⚠ Required variables 'loader' and 'tracker' not found")
    print("This notebook should be run from main_pipeline.ipynb or after loading data")

Initializing baseline experiments with real TCGA data...
⚠ Required variables 'loader' and 'tracker' not found
This notebook should be run from main_pipeline.ipynb or after loading data
